# Testing out the titiler endpoints with some viz

Inspired by the [titiler-cmr notebook](https://github.com/developmentseed/titiler-cmr/blob/develop/docs/examples/xarray_backend_example.ipynb).

In [3]:
import json
from datetime import datetime, timezone

import httpx
import xarray as xr
from folium import Map, TileLayer

titiler_endpoint = "http://localhost:8000"  # running locally accordig to tt-multidim Readme
variable = 'Rainf'
dataset_url = 'tests/fixtures/nldas3_subsampled.zarr'

In [7]:
datetime_ = datetime(2001, 1, 2, tzinfo=timezone.utc).isoformat()

In [8]:
r = httpx.get(
    f"{titiler_endpoint}/WebMercatorQuad/tilejson.json",
    params=(
        ("url", dataset_url),
        # Datetime in form of `start_date/end_date`
        ("datetime", datetime_),
        # titiler-cmr can work with both Zarr and COG dataset
        # but we need to tell the endpoints in advance which backend
        # to use
        ("backend", "xarray"),
        ("variable", variable),
        # We need to set min/max zoom because we don't want to use lowerzoom level (e.g 0)
        # which will results in useless large scale query
        ("minzoom", 2),
        ("maxzoom", 13),
        ("rescale", "0,1"),
        ("colormap_name", "blues_r"),
    ),
).json()

print(r)


{'tilejson': '2.2.0', 'version': '1.0.0', 'scheme': 'xyz', 'tiles': ['http://localhost:8000/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?url=tests%2Ffixtures%2Fnldas3_subsampled.zarr&datetime=2001-01-02T00%3A00%3A00%2B00%3A00&backend=xarray&variable=Rainf&rescale=0%2C1&colormap_name=blues_r'], 'minzoom': 2, 'maxzoom': 13, 'bounds': [-74.49999510639846, 40.500001078904276, -73.49999726420701, 41.499998921095724], 'center': [-73.99999618530273, 41.0, 2]}


In [9]:
bounds = r["bounds"]
m = Map(location=(70, -40), zoom_start=3)

TileLayer(
    tiles=r["tiles"][0],
    opacity=1,
    attr="NASA",
).add_to(m)
m

